In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


In [2]:
!python /content/sample_data/EXP_STEP1_sft.py

Loading glaive-function-calling-v2 dataset ...
README.md: 100% 106/106 [00:00<00:00, 378kB/s]
glaive-function-calling-v2.json: 100% 271M/271M [00:02<00:00, 118MB/s]
Generating train split: 100% 112960/112960 [00:02<00:00, 47138.43 examples/s]
Total examples in dataset: 112960
Parsing dataset examples ...
config.json: 100% 660/660 [00:00<00:00, 3.28MB/s]
tokenizer_config.json: 7.30kB [00:00, 3.15MB/s]
vocab.json: 2.78MB [00:01, 2.62MB/s]
merges.txt: 1.67MB [00:00, 1.98MB/s]
tokenizer.json: 7.03MB [00:02, 2.43MB/s]
Parsed 500 examples  |  Skipped 0
Train: 450  |  Eval: 50

Loading tokenizer ...
Loading model (Qwen2.5-1.5B) ...
model.safetensors: 100% 3.09G/3.09G [00:26<00:00, 118MB/s]
Loading weights: 100% 338/338 [00:10<00:00, 31.58it/s, Materializing param=model.norm.weight] 
generation_config.json: 100% 242/242 [00:00<00:00, 1.23MB/s]
GPU available: True  |  Device: Tesla T4
Adding EOS to train dataset: 100% 450/450 [00:00<00:00, 15103.72 examples/s]
Tokenizing train dataset: 100% 450

In [4]:
!python /content/sample_data/EXP_STEP2_grpo.py

Loading glaive dataset for GRPO ...
Building GRPO prompts ...
GRPO dataset size: 200

Loading tokenizer ...
Loading base model ...
Loading weights: 100% 338/338 [00:10<00:00, 32.78it/s, Materializing param=model.norm.weight] 
Loading SFT adapter from /content/qwen15b-glaive-sft ...
Model ready for GRPO.

🚀 Starting GRPO (RL) training ...

What GRPO does each step:
  1. Generate 4 different tool call candidates per prompt
  2. Score each with reward functions above
  3. Update model to prefer higher-scoring outputs

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
  0% 0/100 [00:00<?, ?it/s]Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_co

In [7]:
!python /content/sample_data/EXP_STEP3_compare.py


Loading SFT model ...
Loading weights: 100% 338/338 [00:01<00:00, 235.50it/s, Materializing param=model.norm.weight]
  SFT model ready.

Loading GRPO model ...
Loading weights: 100% 338/338 [00:01<00:00, 273.53it/s, Materializing param=model.norm.weight]
  GRPO model ready.

                   SIDE-BY-SIDE COMPARISON: SFT vs GRPO (RL)                    

[01] What's the weather in Tokyo?
  Expected tool : get_current_weather | args: ['location']
  SFT  [0.00] : I'm sorry, but as an AI, I don't have real-time data or capabilities to provide current weather cond
  GRPO [0.30] : {"name": "weather_report", "arguments": { "city": "Tokyo" }}
  Winner → GRPO

[02] Is it raining in London right now?
  Expected tool : get_current_weather | args: ['location']
  SFT  [0.00] : I'm sorry, but as an AI, I don't have real-time data capabilities. My responses are based on histori
  GRPO [0.30] : {"name": "weather_api", "arguments": { "city": "London", "action": "is_raining" }}
  Winner → GRPO

[03] 